In [1]:

import re

INPUT_FILE = "Desktop/barca-hackathon/BAR-ATH_tactical_atd.txt"
OUTPUT_FILE = "Desktop/barca-hackathon/BAR-ATH_tactical_atd_txt_converted_data.json"

CHUNK_SIZE = 5 * 1024 * 1024  # 5 MB


NUMBER_PATTERN = re.compile(
    r'-?(?:0|[1-9]\d*)(?:\.\d+)?(?:[eE][+-]?\d+)?$'
)


def format_value(value):
    value = value.strip()

    if value == "NaN":
        return "NaN"

    if NUMBER_PATTERN.fullmatch(value):
        return value

    # Preserve other values as strings
    import json
    return json.dumps(value)


def convert_txt_to_json(input_file, output_file, chunk_size=CHUNK_SIZE):

    record_count = 0
    buffer = ""

    with open(input_file, "r", encoding="utf-8-sig") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:

        outfile.write("[\n")
        first_record = True

        def process_record(record):

            nonlocal record_count, first_record

            record = record.strip()

            if not record:
                return

            if ":" in record:
                frame_id, values = record.split(":", 1)
                frame_id = format_value(frame_id)
            else:
                frame_id = '""'
                values = record

            parts = values.split(",")

            if len(parts) != 3:
                raise ValueError(
                    f"Invalid record #{record_count + 1}: {record}"
                )

            x, y, speed = [
                format_value(value) for value in parts
            ]

            json_record = (
                f'    {{"frame_id": {frame_id}, '
                f'"x": {x}, '
                f'"y": {y}, '
                f'"speed": {speed}}}'
            )

            if not first_record:
                outfile.write(",\n")

            outfile.write(json_record)

            first_record = False
            record_count += 1

        while True:

            chunk = infile.read(chunk_size)

            if not chunk:
                break

            buffer += chunk
            records = buffer.split(";")

            buffer = records.pop()

            for record in records:
                process_record(record)

        # Process final record
        if buffer.strip():
            process_record(buffer)

        outfile.write("\n]\n")

    print("Conversion completed successfully!")
    print(f"Total records converted: {record_count}")
    print(f"Output file: {output_file}")



convert_txt_to_json(INPUT_FILE, OUTPUT_FILE)


ValueError: Invalid record #46: 0.06057,0.39965,NaN:NaN,NaN,NaN
2:0.33404,0.90487,0.8868629932403564

In [2]:

import re
import json



INPUT_FILE = "Desktop/barca-hackathon/BAR-ATH_tactical_atd.txt"
OUTPUT_FILE = "Desktop/barca-hackathon/BAR-ATH_tactical_atd_txt_converted_data.json"


CHUNK_SIZE = 5 * 1024 * 1024



NUMBER_PATTERN = re.compile(
    r'-?(?:0|[1-9]\d*)(?:\.\d+)?(?:[eE][+-]?\d+)?$'
)


def format_value(value):
    value = value.strip()

    if value == "NaN":
        return "NaN"

    if NUMBER_PATTERN.fullmatch(value):
        return value

    return json.dumps(value)


def convert_txt_to_json(input_file, output_file, chunk_size=CHUNK_SIZE):

    record_count = 0
    buffer = ""

    with open(input_file, "r", encoding="utf-8-sig") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:

        outfile.write("[\n")
        first_record = True

        def write_object(frame_id, values):

            nonlocal record_count, first_record

            parts = values.split(",")

            if len(parts) != 3:
                raise ValueError(
                    f"Invalid coordinate group: {values}"
                )

            x, y, speed = [
                format_value(value) for value in parts
            ]

            json_record = (
                f'    {{"frame_id": {frame_id}, '
                f'"x": {x}, '
                f'"y": {y}, '
                f'"speed": {speed}}}'
            )

            if not first_record:
                outfile.write(",\n")

            outfile.write(json_record)

            first_record = False
            record_count += 1

        def process_record(record):

            record = record.strip()

            if not record:
                return

            # Separate frame ID from coordinate groups
            if ":" in record:

                segments = record.split(":")

                # First segment is frame ID if it is an integer
                if re.fullmatch(r'\s*\d+\s*', segments[0]):
                    frame_id = format_value(segments[0])
                    coordinate_groups = segments[1:]
                else:
                    frame_id = '""'
                    coordinate_groups = segments

            else:
                frame_id = '""'
                coordinate_groups = [record]

            for group in coordinate_groups:

                group = group.strip()

                if group:
                    write_object(frame_id, group)

        # Read file in chunks
        while True:

            chunk = infile.read(chunk_size)

            if not chunk:
                break

            buffer += chunk

            records = buffer.split(";")
            buffer = records.pop()

            for record in records:
                process_record(record)

        # Process remaining data
        if buffer.strip():
            process_record(buffer)

        outfile.write("\n]\n")

    print("Conversion completed successfully!")
    print(f"Total JSON objects created: {record_count}")
    print(f"Output saved to: {output_file}")



convert_txt_to_json(INPUT_FILE, OUTPUT_FILE)


Conversion completed successfully!
Total JSON objects created: 9367993
Output saved to: Desktop/barca-hackathon/BAR-ATH_tactical_atd_txt_converted_data.json


In [ ]:
| Expected tracked objects | Around 22 during play |
| Average tracked objects | 20.5 per graded frame |
| Occasional extra objects | Yes, tracker duplicates can push the count above 22 |
| Frame rate | 25 FPS |
| Coordinates | Normalized |

This is enough to design the basic parser and frame-timing logic. One important detail: we should not assume there are exactly 22 objects in every frame. The parser needs to support a variable number of observations and preserve missing values.
At 25 FPS, the time between frames is:

Delta t =  0.04 seconds

So, if the first frame has timestamp zero, frame n has timestamp:

t_n = {n-1}/25

There’s an important possibility here: your data resembles Metrica Sports Automatic Tracking Data (ATD). That format uses 46 track slots even though only about 20–22 objects are typically visible. The slots are not necessarily stable player identities: tracks can disappear, reappear, or switch between people. So we should treat them as tracker slots, not shirt numbers or permanent player IDs.
Kloppy may already support your exact format
Kloppy 3.19.1 has a dedicated loader for Metrica EPTS tracking files:

from kloppy import metrica

dataset = metrica.load_tracking_epts(
    meta_data="Desktop/barca-hackathon/BAR-ATH_tactical_atd.xml",
    raw_data="Desktop/barca-hackathon/BAR-ATH_tactical_atd.txt",
)

print(dataset)

It requires two files:
- tracking.txt — your raw tracking data.(BAR-ATH_tactical_atd.txt)
- metadata.xml — the corresponding match metadata.(BAR-ATH_tactical_atd.xml)
The official documentation includes this exact loading method.

print("Frames:", len(dataset.frames))
print("Frame rate:", dataset.metadata.frame_rate)
print("Pitch dimensions:", dataset.metadata.pitch_dimensions)

# Inspect the first frame
frame = dataset.frames[0]

print("Frame ID:", frame.frame_id)
print("Timestamp:", frame.timestamp)
print("Number of tracked players:", len(frame.players_coordinates))
print("Ball:", frame.ball_coordinates)

You can also convert it into a Pandas DataFrame:
df = dataset.to_df(engine="pandas")

display(df.head())

In [3]:
#!pip install kloppy
from kloppy import metrica
import pandas 

dataset = metrica.load_tracking_epts(
    meta_data="Desktop/barca-hackathon/BAR-ATH_tactical_atd.xml",
    raw_data="Desktop/barca-hackathon/BAR-ATH_tactical_atd.txt",
)

print(dataset)


print("Frames:", len(dataset.frames))
print("Frame rate:", dataset.metadata.frame_rate)
print("Pitch dimensions:", dataset.metadata.pitch_dimensions)

# Inspect the first frame
frame = dataset.frames[0]

print("Frame ID:", frame.frame_id)
print("Timestamp:", frame.timestamp)
print("Number of tracked players:", len(frame.players_coordinates))
print("Ball:", frame.ball_coordinates)


df = dataset.to_df(engine="pandas")

display(df.head())

<TrackingDataset record_count=199319>
Frames: 199319
Frame rate: 25
Pitch dimensions: NormalizedPitchDimensions(x_dim=Dimension(min=0, max=1), y_dim=Dimension(min=0, max=1), standardized=False, unit=<Unit.NORMED: 'normed'>, goal_width=0.10764705882352942, goal_height=2.44, six_yard_width=0.26941176470588235, six_yard_length=0.05238095238095238, penalty_area_width=0.5929411764705882, penalty_area_length=0.15714285714285714, circle_radius=0.08714285714285715, corner_radius=0.009523809523809525, penalty_spot_distance=0.10476190476190476, penalty_arc_radius=0.08714285714285715, pitch_length=105, pitch_width=68)
Frame ID: 1
Timestamp: 0:00:00.040000
Number of tracked players: 46
Ball: Point3D(x=nan, y=nan, z=None)


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,44_d,44_s,45_x,45_y,45_d,45_s,46_x,46_y,46_d,46_s
0,1,0 days 00:00:00.040000,1,None,None,NaN,NaN,None,None,0.33411,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
1,1,0 days 00:00:00.080000,2,None,None,NaN,NaN,None,None,0.33404,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
2,1,0 days 00:00:00.120000,3,None,None,NaN,NaN,None,None,0.33396,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
3,1,0 days 00:00:00.160000,4,None,None,NaN,NaN,None,None,0.33389,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
4,1,0 days 00:00:00.200000,5,None,None,NaN,NaN,None,None,0.33382,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN


In [4]:
df.to_csv("Desktop/barca-hackathon/BAR-ATH_tactical_atd.csv", index=False)

In [5]:
broadcast_dataset = metrica.load_tracking_epts(
    meta_data="Desktop/barca-hackathon/broadcast/BAR-ATH_broadcast_atd.xml",
    raw_data="Desktop/barca-hackathon/broadcast/BAR-ATH_broadcast_atd.txt",
)

print(broadcast_dataset)


print("Frames:", len(broadcast_dataset.frames))
print("Frame rate:", broadcast_dataset.metadata.frame_rate)
print("Pitch dimensions:", broadcast_dataset.metadata.pitch_dimensions)

# Inspect the first frame
frame = broadcast_dataset.frames[0]

print("Frame ID:", frame.frame_id)
print("Timestamp:", frame.timestamp)
print("Number of tracked players:", len(frame.players_coordinates))
print("Ball:", frame.ball_coordinates)


broadcast_df = broadcast_dataset.to_df(engine="pandas")

display(broadcast_df.head())
broadcast_df.to_csv("Desktop/barca-hackathon/BAR-ATH_broadcast_atd.csv", index=False)

<TrackingDataset record_count=164488>
Frames: 164488
Frame rate: 25
Pitch dimensions: NormalizedPitchDimensions(x_dim=Dimension(min=0, max=1), y_dim=Dimension(min=0, max=1), standardized=False, unit=<Unit.NORMED: 'normed'>, goal_width=0.10764705882352942, goal_height=2.44, six_yard_width=0.26941176470588235, six_yard_length=0.05238095238095238, penalty_area_width=0.5929411764705882, penalty_area_length=0.15714285714285714, circle_radius=0.08714285714285715, corner_radius=0.009523809523809525, penalty_spot_distance=0.10476190476190476, penalty_arc_radius=0.08714285714285715, pitch_length=105, pitch_width=68)
Frame ID: 1
Timestamp: 0:00:00.040000
Number of tracked players: 51
Ball: Point3D(x=nan, y=nan, z=None)


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,49_d,49_s,50_x,50_y,50_d,50_s,51_x,51_y,51_d,51_s
0,1,0 days 00:00:00.040000,1,None,None,NaN,NaN,None,None,NaN,...,None,NaN,NaN,NaN,None,NaN,0.02469,0.4836,None,NaN
1,1,0 days 00:00:00.080000,2,None,None,NaN,NaN,None,None,NaN,...,None,NaN,NaN,NaN,None,NaN,0.02469,0.4836,None,NaN
2,1,0 days 00:00:00.120000,3,None,None,NaN,NaN,None,None,NaN,...,None,NaN,NaN,NaN,None,NaN,0.02469,0.4836,None,NaN
3,1,0 days 00:00:00.160000,4,None,None,NaN,NaN,None,None,NaN,...,None,NaN,NaN,NaN,None,NaN,0.02469,0.4836,None,NaN
4,1,0 days 00:00:00.200000,5,None,None,NaN,NaN,None,None,NaN,...,None,NaN,NaN,NaN,None,NaN,0.02469,0.4836,None,NaN


In [1]:
from kloppy import sportscode

events = sportscode.load(
    "Desktop/barca-hackathon/tactical/BAR-ATH_tactical_smart_tagging.xml"
)

print(events)
events_df = events.to_df(engine="pandas")

display(events_df.head())

<CodeDataset record_count=638>


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,None,None
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,None,None


In [2]:
events_df.to_csv("Desktop/barca-hackathon/tactical/BAR-ATH_tactical_smart_tagging.csv", index=False)

In [6]:
import pandas as pd
import numpy as np

# Load CSV
#df = pd.read_csv("smart_tagging.csv")

FPS = 25
FRAME_DURATION = pd.Timedelta(seconds=1 / FPS)

# Convert timestamps into Pandas Timedelta
events_df["timestamp"] = pd.to_timedelta(events_df["timestamp"])
events_df["end_timestamp"] = pd.to_timedelta(events_df["end_timestamp"])

expanded_rows = []

# Expand events into individual frames
for _, row in events_df.iterrows():

    start = row["timestamp"]
    end = row["end_timestamp"]

    frame_timestamps = pd.timedelta_range(
    start=start,
    end=end,
    freq=FRAME_DURATION
)

# Exclude the end timestamp (left-inclusive, right-exclusive)
    frame_timestamps = frame_timestamps[
        frame_timestamps < end
]

    for timestamp in frame_timestamps:

        new_row = row.copy()
        new_row["timestamp"] = timestamp

        expanded_rows.append(new_row)

# Create expanded DataFrame
expanded_df = pd.DataFrame(expanded_rows)

# ------------------------------------------------
# MERGE OVERLAPPING EVENTS
# ------------------------------------------------

# Columns that should not be aggregated
key_columns = ["timestamp"]

# All remaining columns are event attributes
value_columns = [
    col for col in expanded_df.columns
    if col not in key_columns + ["end_timestamp"]
]

def merge_values(series):
    """
    Merge values:
    - Sum different numerical values.
    - Preserve identical numerical values.
    - Combine distinct text values.
    - Ignore missing values.
    """

    values = series.dropna().unique()

    if len(values) == 0:
        return np.nan

    if len(values) == 1:
        return values[0]

    # Numerical values: add them
    if pd.api.types.is_numeric_dtype(series):
        return series.sum()

    # Text values: combine distinct values
    return " | ".join(map(str, values))


# Group events by frame timestamp
merged_df = (
    expanded_df
    .groupby("timestamp", as_index=False)
    .agg({
        **{
            col: merge_values
            for col in value_columns
        }
    })
)

# Sort chronologically
merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)

# Save CSV
merged_df.to_csv("Desktop/barca-hackathon/tactical/BAR-ATH_tactical_smart_taggingsmart_25fps.csv", index=False)

# Display results
display(merged_df.head(20))

print("Original rows:", len(df))
print("Expanded frames before merging:", len(expanded_df))
print("Frames after merging:", len(merged_df))

,timestamp,code_id,period_id,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,0 days 00:07:22.760000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
1,0 days 00:07:22.800000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
2,0 days 00:07:22.840000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
3,0 days 00:07:22.880000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
4,0 days 00:07:22.920000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
5,0 days 00:07:22.960000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
6,0 days 00:07:23,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
7,0 days 00:07:23.040000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
8,0 days 00:07:23.080000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN
9,0 days 00:07:23.120000,104,1,BALL IN FINAL THIRD,Atletic Club,NaN,1st Half,4,Central,NaN


NameError: name 'df' is not defined

In [7]:

print("Expanded frames before merging:", len(expanded_df))
print("Frames after merging:", len(merged_df))

Expanded frames before merging: 409697
Frames after merging: 140298


In [5]:
import pandas as pd

# ==========================================
# CONFIGURATION
# ==========================================

RELATIVE_FILE = "./data/tactical/BAR-ATH_tactical_atd.csv"
ABSOLUTE_FILE = "./data/tactical/tagit12.csv"
OUTPUT_FILE = "./data/tactical/merged_dataset.csv"

ABSOLUTE_TIMESTAMP_COL = "frameTimestamp"
RELATIVE_TIMESTAMP_COL = "timestamp"

# ==========================================
# LOAD CSV FILES
# ==========================================

df_abs = pd.read_csv(ABSOLUTE_FILE)
df_rel = pd.read_csv(RELATIVE_FILE)

# Preserve original column ordering
abs_columns = list(df_abs.columns)
rel_columns = list(df_rel.columns)

# ==========================================
# CONVERT TIMESTAMPS
# ==========================================

df_abs["_merge_time"] = pd.to_timedelta(
    df_abs[ABSOLUTE_TIMESTAMP_COL]
)

df_rel["_relative_time"] = pd.to_timedelta(
    df_rel[RELATIVE_TIMESTAMP_COL]
)

# ==========================================
# AUTOMATICALLY CALCULATE OFFSET
# ==========================================

# First absolute timestamp is the starting point
absolute_start = df_abs["_merge_time"].min()

# Add absolute starting timestamp to relative timestamps
df_rel["_merge_time"] = (
    absolute_start + df_rel["_relative_time"]
)

print("Absolute starting timestamp:", absolute_start)
print("Relative starting timestamp:", df_rel["_relative_time"].min())

# ==========================================
# MERGE DATASETS
# ==========================================

merged_df = pd.merge(
    df_abs,
    df_rel,
    on="_merge_time",
    how="outer",
    suffixes=("_absolute", "_relative")
)

# ==========================================
# CHRONOLOGICAL SORTING
# ==========================================

merged_df = (
    merged_df
    .sort_values("_merge_time")
    .reset_index(drop=True)
)

# ==========================================
# COLUMN ORDERING
# ==========================================

final_columns = []

# Absolute CSV columns first
for col in abs_columns:
    if col in merged_df.columns:
        final_columns.append(
            col + "_absolute" if col in rel_columns else col
        )

# Relative CSV columns second
for col in rel_columns:
    if col == RELATIVE_TIMESTAMP_COL:
        continue

    if col in merged_df.columns:
        final_columns.append(
            col + "_relative" if col in abs_columns else col
        )

# Include relative timestamp for reference
final_columns.append(RELATIVE_TIMESTAMP_COL)

merged_df = merged_df[final_columns]

# ==========================================
# EXPORT
# ==========================================

merged_df.to_csv(OUTPUT_FILE, index=False)

# ==========================================
# SUMMARY
# ==========================================

print("\n========== MERGE SUMMARY ==========")

print("Absolute CSV rows:", len(df_abs))
print("Relative CSV rows:", len(df_rel))
print("Merged CSV rows:", len(merged_df))

print("\nOutput columns:")
print(merged_df.columns.tolist())

print("\nMerged dataset preview:")
display(merged_df.head(20))

print("\nOutput saved to:", OUTPUT_FILE)

Absolute starting timestamp: 0 days 00:07:22.760000
Relative starting timestamp: 0 days 00:00:00.040000

========== MERGE SUMMARY ==========
Absolute CSV rows: 139587
Relative CSV rows: 199319
Merged CSV rows: 199320

Output columns:
['Half', 'FrameIndex', 'frameTimestamp', 'Frame Duration', 'Category 1: Circuit Breakers', 'Category 2: Offensive Phases', 'Category 3: Defensive Phases', 'Category 4: Tactical Noise & Spatial', 'frame_id', 'ball_state', 'ball_owning_team_id', 'ball_x', 'ball_y', 'ball_z', 'ball_speed', '1_x', '1_y', '1_d', '1_s', '2_x', '2_y', '2_d', '2_s', '3_x', '3_y', '3_d', '3_s', '4_x', '4_y', '4_d', '4_s', '5_x', '5_y', '5_d', '5_s', '6_x', '6_y', '6_d', '6_s', '7_x', '7_y', '7_d', '7_s', '8_x', '8_y', '8_d', '8_s', '9_x', '9_y', '9_d', '9_s', '10_x', '10_y', '10_d', '10_s', '11_x', '11_y', '11_d', '11_s', '12_x', '12_y', '12_d', '12_s', '13_x', '13_y', '13_d', '13_s', '14_x', '14_y', '14_d', '14_s', '15_x', '15_y', '15_d', '15_s', '16_x', '16_y', '16_d', '16_s', '1

,Half,FrameIndex,frameTimestamp,Frame Duration,Category 1: Circuit Breakers,Category 2: Offensive Phases,Category 3: Defensive Phases,Category 4: Tactical Noise & Spatial,frame_id,ball_state,...,44_s,45_x,45_y,45_d,45_s,46_x,46_y,46_d,46_s,timestamp
0,1st Half,11069.0,00:07:22.760,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1st Half,11070.0,00:07:22.800,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.040000
2,1st Half,11071.0,00:07:22.840,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.080000
3,1st Half,11072.0,00:07:22.880,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,3.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.120000
4,1st Half,11073.0,00:07:22.920,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.160000
5,1st Half,11074.0,00:07:22.960,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.200000
6,1st Half,11075.0,00:07:23.000,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.240000
7,1st Half,11076.0,00:07:23.040,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,7.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.280000
8,1st Half,11077.0,00:07:23.080,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,8.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.320000
9,1st Half,11078.0,00:07:23.120,30.00:07:23,NaN,NaN,NaN,Atletic Club : BALL IN FINAL THIRD : Type=NaN ...,9.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.06057,0.39965,NaN,NaN,0 days 00:00:00.360000



Output saved to: ./data/tactical/merged_dataset.csv


In [2]:
import os
import sys
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURATION
# ==============================================================================
MERGED_FILE = "./data/tactical/merged_tactical_tagged.csv"
EXPECTED_TOTAL_ROWS = 199319
KICKOFF_FRAME_EST = 11050  # ~00:07:22 (5 seconds before kickoff)

TAG_COLS = [
    "Half",
    "Frame Timestamp",
    "Frame Duration",
    "Category 1: Circuit Breakers",
    "Category 2: Offensive Phases",
    "Category 3: Defensive Phases",
    "Category 4: Tactical Noise & Spatial"
]

def run_validation():
    print("=" * 75)
    print("POST-MERGE AUTOMATED VALIDATION SUITE")
    print(f"Target File: {MERGED_FILE}")
    print("=" * 75)

    if not os.path.exists(MERGED_FILE):
        print(f"[FAIL] Target file not found: {MERGED_FILE}")
        sys.exit(1)

    print("\nLoading merged dataset for validation...")
    df = pd.read_csv(MERGED_FILE, low_memory=False)
    total_rows, total_cols = df.shape
    print(f"Loaded {total_rows:,} rows and {total_cols} columns.\n")

    all_passed = True

    # --------------------------------------------------------------------------
    # CHECK 1: Total Row Count (Competition Rule Check)
    # --------------------------------------------------------------------------
    print("--- CHECK 1: Row Count Rule (Master Tactical Reference) ---")
    if total_rows == EXPECTED_TOTAL_ROWS:
        print(f"  [PASS] Exactly {total_rows:,} rows found. Matches master timeline.")
    elif total_rows in [164488, 164487, 164489]:
        print(f"  [CRITICAL FAIL] Found ~164k rows ({total_rows:,}).")
        print("                  You indexed the BROADCAST feed instead of Tactical!")
        all_passed = False
    else:
        print(f"  [FAIL] Row count mismatch! Expected {EXPECTED_TOTAL_ROWS:,}, got {total_rows:,}.")
        all_passed = False

    # --------------------------------------------------------------------------
    # CHECK 2: Frame Index Continuity & Duplicates
    # --------------------------------------------------------------------------
    print("\n--- CHECK 2: Frame ID Continuity & Monotonicity ---")
    if "frame_id" not in df.columns:
        print("  [FAIL] 'frame_id' column is missing from the merged file.")
        all_passed = False
    else:
        frame_series = pd.to_numeric(df["frame_id"], errors="coerce")
        n_null_frames = frame_series.isna().sum()
        n_duplicates = frame_series.duplicated().sum()
        is_monotonic = frame_series.is_monotonic_increasing
        min_frame, max_frame = frame_series.min(), frame_series.max()

        if n_null_frames == 0 and n_duplicates == 0 and is_monotonic:
            print(f"  [PASS] Frame IDs are strictly increasing from {min_frame} to {max_frame} with 0 duplicates/nulls.")
        else:
            print(f"  [FAIL] Frame ID anomalies detected: Nulls={n_null_frames}, Duplicates={n_duplicates}, Monotonic={is_monotonic}")
            all_passed = False

    # --------------------------------------------------------------------------
    # CHECK 3: Pre-Kickoff Inactive Window (< Frame 11,050 / 00:07:22)
    # --------------------------------------------------------------------------
    print(f"\n--- CHECK 3: Pre-Kickoff Boundary Check (Frames 1 to {KICKOFF_FRAME_EST}) ---")
    present_tag_cols = [c for c in TAG_COLS if c in df.columns]
    
    if not present_tag_cols:
        print("  [FAIL] No Smart Tagging columns found in the merged file.")
        all_passed = False
    else:
        df_pre = df[df["frame_id"] < KICKOFF_FRAME_EST]
        leakage_detected = False

        for col in present_tag_cols:
            # Check for non-NaN / non-'NaN' values
            series = df_pre[col].astype(str).str.strip().str.upper()
            non_nan_count = (~series.isin(["NAN", "NONE", "", "NULL"])).sum()
            if non_nan_count > 0:
                print(f"  [FAIL] Column '{col}' has {non_nan_count} non-NaN values before kickoff frame {KICKOFF_FRAME_EST}!")
                leakage_detected = True

        if not leakage_detected:
            print(f"  [PASS] All {len(present_tag_cols)} tagging columns are cleanly filled with 'NaN' prior to frame {KICKOFF_FRAME_EST}.")
        else:
            all_passed = False

    # --------------------------------------------------------------------------
    # CHECK 4: Post-Kickoff Active Tagging Coverage (>= Frame 11,050)
    # --------------------------------------------------------------------------
    print(f"\n--- CHECK 4: Post-Kickoff Event Tagging Coverage ---")
    df_post = df[df["frame_id"] >= KICKOFF_FRAME_EST]
    tagged_summary = {}

    for col in present_tag_cols:
        series = df_post[col].astype(str).str.strip().str.upper()
        active_tags = (~series.isin(["NAN", "NONE", "", "NULL"])).sum()
        pct = (active_tags / len(df_post)) * 100
        tagged_summary[col] = (active_tags, pct)
        print(f"  - {col[:35]:<35} : {active_tags:,} active frames ({pct:.2f}%)")

    # Half column should be populated throughout the match after kickoff
    if "Half" in df.columns:
        half_active = tagged_summary.get("Half", (0, 0))[0]
        if half_active > 0:
            print("  [PASS] Match metadata ('Half') is active post-kickoff.")
        else:
            print("  [FAIL] 'Half' column is completely empty post-kickoff.")
            all_passed = False

    # --------------------------------------------------------------------------
    # CHECK 5: Team Pitch Side Validation (Frame 11,050 / 00:07:22)
    # --------------------------------------------------------------------------
    print(f"\n--- CHECK 5: Tactical Pitch Position Validation at Frame {KICKOFF_FRAME_EST} ---")
    row_ko = df[df["frame_id"] == KICKOFF_FRAME_EST]

    if not row_ko.empty:
        # Check active players in Slots 1-23 (Team 1, Left side X < 0.50)
        team1_x_cols = [f"{i}_x" for i in range(1, 24) if f"{i}_x" in df.columns]
        team1_coords = [pd.to_numeric(row_ko[col].values[0], errors="coerce") for col in team1_x_cols]
        team1_valid = [x for x in team1_coords if pd.notna(x)]

        # Check active outfield players in Slots 24-46 (Team 2, Right side X >= 0.50, excluding anchors)
        team2_x_cols = [f"{i}_x" for i in range(25, 46) if f"{i}_x" in df.columns]
        team2_coords = [pd.to_numeric(row_ko[col].values[0], errors="coerce") for col in team2_x_cols]
        team2_valid = [x for x in team2_coords if pd.notna(x)]

        t1_left = all(x < 0.50 for x in team1_valid) if team1_valid else False
        t2_right_or_center = all(x >= 0.49 for x in team2_valid) if team2_valid else False

        if t1_left and t2_right_or_center:
            print(f"  [PASS] Team 1 (Slots 1-23, n={len(team1_valid)}) on Left (X < 0.50).")
            print(f"  [PASS] Team 2 (Slots 25-45, n={len(team2_valid)}) on Right/Center (X >= 0.49).")
        else:
            print(f"  [WARNING] Spatial distribution anomaly at frame {KICKOFF_FRAME_EST}.")
            print(f"            Team 1 Left: {t1_left}, Team 2 Right: {t2_right_or_center}")
    else:
        print(f"  [WARNING] Frame {KICKOFF_FRAME_EST} could not be inspected directly.")

    # --------------------------------------------------------------------------
    # CHECK 6: Coordinate Integrity Check (No Out-of-Bounds or String Pollution)
    # --------------------------------------------------------------------------
    print("\n--- CHECK 6: Spatial Tracking Coordinates Health ---")
    coord_sample_cols = [c for c in ["ball_x", "1_x", "24_x"] if c in df.columns]
    corrupted_coords = 0

    for col in coord_sample_cols:
        num_series = pd.to_numeric(df[col].replace("NaN", np.nan), errors="coerce")
        # Standard pitch coordinates normalized to [0, 1]
        out_of_bounds = ((num_series < -0.1) | (num_series > 1.1)).sum()
        if out_of_bounds > 0:
            print(f"  [FAIL] {col} has {out_of_bounds} coordinates outside valid pitch bounds.")
            corrupted_coords += 1

    if corrupted_coords == 0:
        print(f"  [PASS] Coordinate ranges for tracking columns are valid and uncorrupted.")
    else:
        all_passed = False

    # --------------------------------------------------------------------------
    # FINAL VERDICT
    # --------------------------------------------------------------------------
    print("\n" + "=" * 75)
    if all_passed:
        print("OVERALL VERDICT: [SUCCESS] The merged file passes all validation checks.")
        print("                 It is fully aligned and ready for model consumption/submission.")
    else:
        print("OVERALL VERDICT: [FAILED] One or more critical checks failed.")
        print("                 Review the error logs above before submitting.")
    print("=" * 75)

import contextlib
from datetime import datetime



import contextlib
import traceback
from datetime import datetime



output_file = f"./data/tactical/validation_results_with_errors{datetime.now():%Y%m%d_%H%M%S}.txt"

with open(output_file, "w", encoding="utf-8") as f:

    with contextlib.redirect_stdout(f), \
         contextlib.redirect_stderr(f):

        try:
            run_validation()

        except Exception:
            traceback.print_exc()

print(f"Results saved to: {output_file}")

Results saved to: ./data/tactical/validation_results_with_errors20261114_000104.txt


In [8]:
import numpy as np
import pandas as pd
from typing import Tuple

# =====================================================================
# CONFIGURATION & FILE PATHS (EDIT YOUR CUSTOM NAMES HERE)
# =====================================================================
TACTICAL_TRACKING_PATH    = "./data/tactical/merged_tactical_tagged.csv"      # Master timeline (~199k rows)
TACTICAL_HOMOGRAPHY_PATH  = "./data/tactical/BAR-ATH_tactical_homography.csv"    # Tactical homography matrices / validity flags
BROADCAST_TRACKING_PATH   = "./data/broadcast/BAR-ATH_broadcast_atd.csv"     # Broadcast tracking (~156k–164k rows)
BROADCAST_HOMOGRAPHY_PATH = "./data/broadcast/BAR-ATH_broadcast_homography.csv"   # Broadcast homography / validity flags
OUTPUT_ENRICHED_PATH      = "./data/tactical_enriched.csv"      # Destination CSV for enriched tactical file


# =====================================================================
# 1. ROBUST CSV INGESTION & DATA CLEANING
# =====================================================================
NA_SENTINELS = ['', ' ', '  ', 'None', 'none', 'null', 'NULL', 'NA', 'N/A', '-', '?', 'NaN', 'nan']

def load_clean_csv(file_path: str) -> pd.DataFrame:
    """
    Loads CSV with strict handling for missing/empty values and non-standard placeholders.
    """
    return pd.read_csv(
        file_path,
        na_values=NA_SENTINELS,
        keep_default_na=True,
        low_memory=False
    )


# =====================================================================
# 2. HOMOGRAPHY TRANSFORMATION UTILITIES
# =====================================================================
def apply_planar_homography(
    x_coords: np.ndarray, y_coords: np.ndarray, H_matrices: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Applies per-frame 3x3 planar homography matrices to (x, y) coordinate arrays.
    
    Parameters:
        x_coords: (N, K) array of x-coordinates (K track slots or 1 ball track).
        y_coords: (N, K) array of y-coordinates.
        H_matrices: (N, 3, 3) per-frame transformation matrices.
        
    Returns:
        (x_pitch, y_pitch): Projected metric/normalized pitch coordinates.
    """
    # Ensure float64 array format
    x_coords = np.asarray(x_coords, dtype=np.float64)
    y_coords = np.asarray(y_coords, dtype=np.float64)

    # Broadcast H matrix elements across all K coordinate slots
    # H = [[h00, h01, h02],
    #      [h10, h11, h12],
    #      [h20, h21, h22]]
    h00 = H_matrices[:, 0, 0, np.newaxis]
    h01 = H_matrices[:, 0, 1, np.newaxis]
    h02 = H_matrices[:, 0, 2, np.newaxis]
    h10 = H_matrices[:, 1, 0, np.newaxis]
    h11 = H_matrices[:, 1, 1, np.newaxis]
    h12 = H_matrices[:, 1, 2, np.newaxis]
    h20 = H_matrices[:, 2, 0, np.newaxis]
    h21 = H_matrices[:, 2, 1, np.newaxis]
    h22 = H_matrices[:, 2, 2, np.newaxis]

    # Homogeneous transformation: [X, Y, W]^T = H @ [x, y, 1]^T
    W = h20 * x_coords + h21 * y_coords + h22
    # Guard against division by zero (e.g., degenerate homographies)
    W_safe = np.where(np.abs(W) < 1e-7, np.nan, W)

    X_proj = (h00 * x_coords + h01 * y_coords + h02) / W_safe
    Y_proj = (h10 * x_coords + h11 * y_coords + h12) / W_safe

    return X_proj, Y_proj


def extract_homography_matrices(homog_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extracts (N, 3, 3) homography matrices and a boolean validity mask from the homography file.
    Supports flexible naming conventions: 'valid', 'is_valid', 'status', etc.
    Coerces invalid entries to NaN and creates safe identity fallbacks.
    """
    col_mapping = {col.lower().strip(): col for col in homog_df.columns}
    
    # 1. Resolve validity mask
    valid_col_key = None
    for candidate in ['valid', 'is_valid', 'homography_valid', 'status', 'usable']:
        if candidate in col_mapping:
            valid_col_key = col_mapping[candidate]
            break
            
    if valid_col_key is not None:
        raw_valid = homog_df[valid_col_key]
        if raw_valid.dtype == bool:
            valid_mask = raw_valid.to_numpy()
        else:
            # Handles 1/0, '1'/'0', 'True'/'False', 't'/'f'
            valid_mask = raw_valid.astype(str).str.strip().str.lower().isin(['1', '1.0', 'true', 't']).to_numpy()
    else:
        # If no explicit validity column exists, consider all non-empty frames valid
        valid_mask = np.ones(len(homog_df), dtype=bool)

    # 2. Resolve 3x3 matrix columns
    h_patterns = [
        [f'h{r}{c}' for r in range(3) for c in range(3)],
        [f'h_{r+1}{c+1}' for r in range(3) for c in range(3)],
        [f'h{r+1}{c+1}' for r in range(3) for c in range(3)],
        [f'm{r}{c}' for r in range(3) for c in range(3)]
    ]

    selected_pattern = None
    for pattern in h_patterns:
        if all(p in col_mapping for p in pattern):
            selected_pattern = [col_mapping[p] for p in pattern]
            break

    if selected_pattern is not None:
        # Coerce all matrix elements to float, turning any corrupted/empty cells into NaN
        h_clean = homog_df[selected_pattern].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
        H_matrices = h_clean.reshape(-1, 3, 3)
        # Any frame with a NaN in its matrix should be marked invalid
        has_nan_matrix = np.isnan(H_matrices).any(axis=(1, 2))
        valid_mask = valid_mask & (~has_nan_matrix)
    else:
        # Default fallback to 3x3 identity matrices
        H_matrices = np.tile(np.eye(3, dtype=np.float64), (len(homog_df), 1, 1))

    return H_matrices, valid_mask


# =====================================================================
# 3. VECTORIZED CENTROID & CROSS-CORRELATION ENGINE
# =====================================================================
def compute_vectorized_1d_centroid(df: pd.DataFrame, num_player_slots: int) -> np.ndarray:
    """
    Computes the 1D pitch-x centroid across all active anonymous player track slots
    vectorially, safely ignoring NaNs, empty strings, and out-of-bounds sentinels.
    """
    x_cols = [f'{i}_x' for i in range(1, num_player_slots + 1) if f'{i}_x' in df.columns]
    if not x_cols:
        raise ValueError(f"Could not find any player x columns matching pattern 1_x ... {num_player_slots}_x")

    # Coerce columns to numeric; non-numeric entries become NaN
    player_x_data = df[x_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)

    # Treat out-of-pitch values as NaN
    player_x_data = np.where((player_x_data < -20.0) | (player_x_data > 130.0), np.nan, player_x_data)

    # Compute mean across player axis, handling all-NaN frame rows gracefully
    with np.errstate(invalid='ignore'):
        centroid = np.nanmean(player_x_data, axis=1)

    return centroid


def find_chunk_offset(
    broadcast_segment_centroid: np.ndarray,
    tactical_centroid: np.ndarray,
    search_center: int,
    max_drift_window: int = 500
) -> Tuple[int, float]:
    """
    Performs 1D cross-correlation to find the exact frame shift between
    a live broadcast chunk and the tactical master timeline.
    """
    b_sig = broadcast_segment_centroid
    chunk_len = len(b_sig)

    # Clean signal of internal NaNs via linear interpolation
    valid_b = ~np.isnan(b_sig)
    if np.sum(valid_b) < (0.2 * chunk_len) or np.sum(valid_b) < 10:
        return 0, 0.0
    
    b_clean = np.interp(np.arange(chunk_len), np.flatnonzero(valid_b), b_sig[valid_b])
    b_norm = b_clean - np.mean(b_clean)
    norm_b = np.linalg.norm(b_norm)

    if norm_b < 1e-5:
        return 0, 0.0

    # Constrain search window around estimated timeline position to avoid aliasing replays
    start_search = max(0, search_center - max_drift_window)
    end_search = min(len(tactical_centroid), search_center + chunk_len + max_drift_window)

    t_slice = tactical_centroid[start_search:end_search]
    valid_t = ~np.isnan(t_slice)
    if np.sum(valid_t) < chunk_len:
        return 0, 0.0

    t_clean = np.interp(np.arange(len(t_slice)), np.flatnonzero(valid_t), t_slice[valid_t])
    t_norm = t_clean - np.mean(t_clean)

    # 1D Cross-correlation via FFT-optimized numpy correlate
    corr = np.correlate(t_norm, b_norm, mode='valid')
    if len(corr) == 0:
        return 0, 0.0

    best_idx = int(np.argmax(corr))
    
    # Calculate normalized cross-correlation peak score (-1.0 to 1.0)
    matched_t_norm = t_norm[best_idx : best_idx + chunk_len]
    norm_t = np.linalg.norm(matched_t_norm)
    score = float(corr[best_idx] / (norm_b * norm_t + 1e-9))

    # Global tactical frame offset corresponding to broadcast chunk index 0
    tactical_frame_offset = start_search + best_idx
    return tactical_frame_offset, score


# =====================================================================
# 4. MAIN SYNCHRONIZATION & SURGICAL ENRICHMENT PIPELINE
# =====================================================================
def run_broadcast_synchronization(
    tactical_tracking_path: str,
    tactical_homography_path: str,
    broadcast_tracking_path: str,
    broadcast_homography_path: str,
    output_enriched_path: str,
    min_chunk_duration_frames: int = 75,   # Minimum continuous live-play window (~3.0 seconds)
    min_correlation_confidence: float = 0.65
) -> pd.DataFrame:
    """
    Main orchestration function to homogenize, align, and inject broadcast ball tracking
    into the master tactical timeline.
    """
    print("[1/5] Loading raw tracking and separate homography data...")
    tactical_df = load_clean_csv(tactical_tracking_path)
    tactical_homog_df = load_clean_csv(tactical_homography_path)
    broadcast_df = load_clean_csv(broadcast_tracking_path)
    broadcast_homog_df = load_clean_csv(broadcast_homography_path)

    print(f"      Tactical master frames: {len(tactical_df):,}")
    print(f"      Broadcast frames:      {len(broadcast_df):,}")

    print("[2/5] Sanitizing coordinates and processing planar homographies...")
    # Homogenize Tactical feed if present in image/pixel space
    t_H, t_valid = extract_homography_matrices(tactical_homog_df)
    if 'ball_x' in tactical_df.columns:
        tac_bx = pd.to_numeric(tactical_df['ball_x'], errors='coerce').to_numpy(dtype=np.float64)
        tac_by = pd.to_numeric(tactical_df['ball_y'], errors='coerce').to_numpy(dtype=np.float64)
        if np.nanmax(tac_bx) > 150.0:
            bx, by = apply_planar_homography(tac_bx[:, np.newaxis], tac_by[:, np.newaxis], t_H)
            tactical_df['ball_x'] = bx.squeeze()
            tactical_df['ball_y'] = by.squeeze()
        else:
            tactical_df['ball_x'] = tac_bx
            tactical_df['ball_y'] = tac_by

    # Homogenize Broadcast feed
    b_H, b_valid = extract_homography_matrices(broadcast_homog_df)
    if 'ball_x' in broadcast_df.columns:
        broad_bx = pd.to_numeric(broadcast_df['ball_x'], errors='coerce').to_numpy(dtype=np.float64)
        broad_by = pd.to_numeric(broadcast_df['ball_y'], errors='coerce').to_numpy(dtype=np.float64)
        if np.nanmax(broad_bx) > 150.0:
            bx_b, by_b = apply_planar_homography(broad_bx[:, np.newaxis], broad_by[:, np.newaxis], b_H)
            broadcast_df['ball_x'] = bx_b.squeeze()
            broadcast_df['ball_y'] = by_b.squeeze()
        else:
            broadcast_df['ball_x'] = broad_bx
            broadcast_df['ball_y'] = broad_by

    # Mask broadcast tracking using homography validity flag to drop camera cuts and replays
    broadcast_df.loc[~b_valid, ['ball_x', 'ball_y']] = np.nan

    print("[3/5] Extracting 1D spatial signatures (player centroids)...")
    tac_centroid = compute_vectorized_1d_centroid(tactical_df, num_player_slots=46)
    broad_centroid = compute_vectorized_1d_centroid(broadcast_df, num_player_slots=51)

    print("[4/5] Chunking broadcast data into continuous live-play intervals...")
    is_live = b_valid.astype(np.int8)
    diff = np.diff(np.pad(is_live, (1, 1), 'constant'))
    starts = np.flatnonzero(diff == 1)
    ends = np.flatnonzero(diff == -1)

    scale_ratio = len(tactical_df) / max(len(broadcast_df), 1)
    
    injected_ball_count = 0
    tactical_ball_x = pd.to_numeric(tactical_df['ball_x'], errors='coerce').to_numpy(dtype=np.float64)
    tactical_ball_y = pd.to_numeric(tactical_df['ball_y'], errors='coerce').to_numpy(dtype=np.float64)
    broadcast_ball_x = pd.to_numeric(broadcast_df['ball_x'], errors='coerce').to_numpy(dtype=np.float64)
    broadcast_ball_y = pd.to_numeric(broadcast_df['ball_y'], errors='coerce').to_numpy(dtype=np.float64)

    initial_tac_missing = np.sum(np.isnan(tactical_ball_x))
    print(f"      Initial missing ball frames in master timeline: {initial_tac_missing:,}")

    print("[5/5] Aligning segments via 1D cross-correlation and injecting missing ball coordinates...")
    for seg_start, seg_end in zip(starts, ends):
        seg_len = seg_end - seg_start
        if seg_len < min_chunk_duration_frames:
            continue

        b_sub_centroid = broad_centroid[seg_start:seg_end]
        estimated_tac_pos = int(seg_start * scale_ratio)

        tac_offset, confidence = find_chunk_offset(
            b_sub_centroid, tac_centroid, search_center=estimated_tac_pos
        )

        # Discard segments below correlation threshold (e.g. replays, zooms)
        if confidence < min_correlation_confidence:
            continue

        tac_start = tac_offset
        tac_end = tac_offset + seg_len

        # Clamp against master bounds
        if tac_end > len(tactical_df):
            clip = tac_end - len(tactical_df)
            tac_end -= clip
            seg_end -= clip

        if tac_start >= len(tactical_df) or seg_end <= seg_start:
            continue

        # Injection mask: Tactical MUST be NaN AND Broadcast MUST be available
        tac_missing_mask = np.isnan(tactical_ball_x[tac_start:tac_end])
        b_available_mask = ~np.isnan(broadcast_ball_x[seg_start:seg_end])

        inject_mask = tac_missing_mask & b_available_mask

        if np.any(inject_mask):
            tactical_ball_x[tac_start:tac_end] = np.where(
                inject_mask, broadcast_ball_x[seg_start:seg_end], tactical_ball_x[tac_start:tac_end]
            )
            tactical_ball_y[tac_start:tac_end] = np.where(
                inject_mask, broadcast_ball_y[seg_start:seg_end], tactical_ball_y[tac_start:tac_end]
            )
            injected_ball_count += int(np.sum(inject_mask))

    # Save back to DataFrame
    tactical_df['ball_x'] = tactical_ball_x
    tactical_df['ball_y'] = tactical_ball_y

    final_tac_missing = np.sum(np.isnan(tactical_ball_x))
    print(f"\n=======================================================")
    print(f"-> Synchronization Complete!")
    print(f"-> Ball coordinates injected:  {injected_ball_count:,} frames")
    print(f"-> Remaining missing frames:   {final_tac_missing:,} frames")
    print(f"-> Output saved to:            '{output_enriched_path}'")
    print(f"=======================================================\n")

    tactical_df.to_csv(output_enriched_path, index=False)
    return tactical_df


# =====================================================================
# ENTRY POINT
# =====================================================================

run_broadcast_synchronization(
    tactical_tracking_path=TACTICAL_TRACKING_PATH,
    tactical_homography_path=TACTICAL_HOMOGRAPHY_PATH,
    broadcast_tracking_path=BROADCAST_TRACKING_PATH,
    broadcast_homography_path=BROADCAST_HOMOGRAPHY_PATH,
    output_enriched_path=OUTPUT_ENRICHED_PATH
)


[1/5] Loading raw tracking and separate homography data...
      Tactical master frames: 199,319
      Broadcast frames:      164,488
[2/5] Sanitizing coordinates and processing planar homographies...
[3/5] Extracting 1D spatial signatures (player centroids)...
[4/5] Chunking broadcast data into continuous live-play intervals...
      Initial missing ball frames in master timeline: 85,831
[5/5] Aligning segments via 1D cross-correlation and injecting missing ball coordinates...

-> Synchronization Complete!
-> Ball coordinates injected:  5,343 frames
-> Remaining missing frames:   80,488 frames
-> Output saved to:            './data/tactical_enriched.csv'



<ipython-input-8-f2f89ec30469>:152: RuntimeWarning: Mean of empty slice
  centroid = np.nanmean(player_x_data, axis=1)


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,46_y,46_d,46_s,Half,Frame Timestamp,Frame Duration,Category 1: Circuit Breakers,Category 2: Offensive Phases,Category 3: Defensive Phases,Category 4: Tactical Noise & Spatial
0,1,0 days 00:00:00.040000,1,NaN,NaN,NaN,NaN,NaN,NaN,0.33411,...,0.39965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0 days 00:00:00.080000,2,NaN,NaN,NaN,NaN,NaN,NaN,0.33404,...,0.39965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,0 days 00:00:00.120000,3,NaN,NaN,NaN,NaN,NaN,NaN,0.33396,...,0.39965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,0 days 00:00:00.160000,4,NaN,NaN,NaN,NaN,NaN,NaN,0.33389,...,0.39965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,0 days 00:00:00.200000,5,NaN,NaN,NaN,NaN,NaN,NaN,0.33382,...,0.39965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199314,1,0 days 02:12:52.600000,199315,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.51646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
199315,1,0 days 02:12:52.640000,199316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.51646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
199316,1,0 days 02:12:52.680000,199317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.51646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
199317,1,0 days 02:12:52.720000,199318,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.51646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
